# TME 6015 Final Project: which bin does it go in?

Training and evaluation notebook (Ali Rizvi). Run it on Google Colab with a **T4 GPU** (Runtime > Change runtime type). Run all cells in order.

What it does, following the course ladder (start plain, then add one thing at a time):

1. Plain baselines: EfficientNet-B0 on real photos only (B0), the same with synthetic-background images added (B1).
2. Dropout added on its own (D1), then a plain MobileNetV3-Large (B2) and plain ensembles.
3. The full regularization stack (dropout, stochastic depth, label smoothing, weight decay, class weights, augmentation) as a later experiment: E0, E1, E2 and their ensembles.
4. Comparison with published results on the same datasets, the final-model choice (made on the validation sets only), Wilson 95% intervals, a segmentation (isolate-then-classify) experiment, plots, and saving of all models.

The dataset is copied from this repository in cell 2 (`data/trashnet_custom_v2`). The seed is fixed (42), so the numbers in `results/results.csv` are reproduced exactly on the same hardware and library versions. Models are saved to `/content/out` and zipped; set `DOWNLOAD = True` in the last cell to download the zip.

In [ ]:
# === 1. Setup ===
import os, sys, json, glob, random, time, math, zipfile, shutil, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PRETRAINED = True          # ImageNet weights (needs internet; fine on Colab)
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
C2I = {c: i for i, c in enumerate(CLASSES)}
print('torch', torch.__version__, '| device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'NO GPU - change runtime to T4')

In [ ]:
# === 2. Get the dataset ===
# Order: (a) already unpacked in /content/data, (b) copy it from the public GitHub repo, (c) upload TrashNet_custom_v2_dataset.zip by hand.
import subprocess
REPO_URL = 'https://github.com/AliR19/trash-sorting-classifier'
if not os.path.exists('/content/data'):
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/repo'], check=True)
        shutil.copytree('/content/repo/data', '/content/data')
        print('dataset copied from', REPO_URL)
    except Exception as e:
        print('clone failed:', e, '-> upload TrashNet_custom_v2_dataset.zip instead')
        from google.colab import files
        up = files.upload()
        name = list(up.keys())[0]
        with zipfile.ZipFile(name) as z:
            z.extractall('/content/data')
        print('unzipped', name)
for root, dirs, fs in os.walk('/content/data'):
    depth = root[len('/content/data'):].count('/')
    if depth <= 2:
        print(root, '->', len(fs), 'files')

In [ ]:
# === 3. Build the file table: split, class, real or synthetic ===
SPLITS = ['train', 'val', 'test', 'test_external', 'val_bgshift', 'test_bgshift']
rows = []
for p in glob.glob('/content/data/**/*', recursive=True):
    if not p.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    parts = p.split('/')
    cls = next((x for x in reversed(parts[:-1]) if x in C2I), None)
    split = next((x for x in parts[:-1] if x in SPLITS), None)
    if cls is None or split is None:
        continue
    low = p.lower()
    syn = any(t in low for t in ['synth', '/syn', '_syn', 'bgswap', '_bg']) or split.endswith('bgshift')
    rows.append(dict(path=p, split=split, label=C2I[cls], cls=cls, synthetic=syn))
df = pd.DataFrame(rows)
print(len(df), 'images found')
print(df.groupby(['split', 'synthetic']).size().unstack(fill_value=0))
print(df[df.split == 'train'].groupby(['cls', 'synthetic']).size().unstack(fill_value=0))
EXPECTED = {'train': 5030, 'val': 615, 'test': 375, 'test_external': 480, 'val_bgshift': 168, 'test_bgshift': 162}
for s, n in EXPECTED.items():
    got = int((df.split == s).sum())
    print(f'{s:14s} expected {n:5d} got {got:5d}', 'OK' if got == n else '<-- CHECK')
print('synthetic images in train: expected 998, got', int(df[(df.split == 'train') & df.synthetic].shape[0]))
print(df[df.split == 'train'].path.sample(6, random_state=0).tolist())

In [ ]:
# === 4. Data loading, models, training and evaluation helpers ===
H, W = 192, 256            # keeps the 4:3 shape of the photos
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop((H, W), scale=(0.7, 1.0), ratio=(1.2, 1.45)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.03),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
plain_tf = eval_tf          # no augmentation at all (used by the plain baselines)

IMG = {}                    # decoded images are cached in memory so epochs stay fast
def get_img(p):
    if p not in IMG:
        IMG[p] = Image.open(p).convert('RGB').resize((W, H))
    return IMG[p]

class DS(torch.utils.data.Dataset):
    def __init__(self, d, tf):
        self.p, self.y, self.tf = d.path.tolist(), d.label.tolist(), tf
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.tf(get_img(self.p[i])), self.y[i]

def loader(d, tf, shuffle, bs=64):
    for p in d.path:
        get_img(p)
    return torch.utils.data.DataLoader(DS(d, tf), batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

def build(arch, p_drop=0.3, sd=None):
    if arch == 'efficientnet_b0':
        kw = {} if sd is None else dict(stochastic_depth_prob=sd)       # sd=0.0 switches off stochastic depth
        m = models.efficientnet_b0(weights='IMAGENET1K_V1' if PRETRAINED else None, **kw)
        head = [nn.Dropout(p_drop)] if p_drop > 0 else []
        m.classifier = nn.Sequential(*head, nn.Linear(m.classifier[1].in_features, len(CLASSES)))
    elif arch == 'mobilenet_v3_large':
        m = models.mobilenet_v3_large(weights='IMAGENET1K_V2' if PRETRAINED else None)
        m.classifier = nn.Sequential(nn.Linear(m.classifier[0].in_features, 1280), nn.Hardswish(),
                                     nn.Dropout(p_drop), nn.Linear(1280, len(CLASSES)))
    else:
        raise ValueError(arch)
    return m.to(DEVICE)

@torch.no_grad()
def predict(m, dl):
    m.eval()
    out = []
    for x, _ in dl:
        out.append(F.softmax(m(x.to(DEVICE)), 1).float().cpu())
    return torch.cat(out).numpy()

# mode 'plain' = no dropout, no stochastic depth, no label smoothing, no weight decay, no class weights, no augmentation
# mode 'dropout' = plain + dropout in the classifier head only
# mode 'reg' = everything on (dropout, stochastic depth, label smoothing, weight decay, class weights, augmentation)
MODES = {'plain':   dict(p_drop=0.0, sd=0.0, tf='plain', ls=0.0, wd=0.0, weights=False),
         'dropout': dict(p_drop=0.3, sd=0.0, tf='plain', ls=0.0, wd=0.0, weights=False),
         'reg':     dict(p_drop=0.3, sd=None, tf='train', ls=0.1, wd=1e-4, weights=True)}

def train_model(arch, d_train, d_val, epochs=8, warm=2, lr_head=1e-3, lr_ft=3e-4, name='', mode='reg'):
    cfg = MODES[mode]
    torch.manual_seed(SEED)
    m = build(arch, p_drop=cfg['p_drop'], sd=cfg['sd'])
    dl = loader(d_train, train_tf if cfg['tf'] == 'train' else plain_tf, True)
    vl = loader(d_val, eval_tf, False)
    if cfg['weights']:
        cnt = np.bincount(d_train.label.values, minlength=len(CLASSES))
        w = torch.tensor((cnt.sum() / cnt) ** 0.5, dtype=torch.float32, device=DEVICE)
        crit = nn.CrossEntropyLoss(weight=w / w.mean(), label_smoothing=cfg['ls'])
    else:
        crit = nn.CrossEntropyLoss(label_smoothing=cfg['ls'])
    use_amp = DEVICE == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best, best_state, hist, ep = -1, None, [], 0
    for n_ep, freeze, lr in [(warm, True, lr_head), (epochs - warm, False, lr_ft)]:
        for p in m.features.parameters():
            p.requires_grad = not freeze
        opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=cfg['wd'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_ep * len(dl)))
        for _ in range(n_ep):
            m.train(); t0 = time.time(); tl, tc = 0.0, 0
            for x, y in dl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.autocast(device_type='cuda', enabled=use_amp):
                    out = m(x)
                    loss = crit(out, y)
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
                tl += loss.item() * len(y); tc += int((out.argmax(1) == y).sum())
            ep += 1
            vp = predict(m, vl)
            yv = d_val.label.values
            vacc = float((vp.argmax(1) == yv).mean())
            vf1 = float(f1_score(yv, vp.argmax(1), average='macro'))
            vloss = float(-np.log(np.clip(vp[np.arange(len(yv)), yv], 1e-8, 1)).mean())
            hist.append(dict(epoch=ep, loss=tl / len(d_train), train_acc=tc / len(d_train), val_loss=vloss, val_acc=vacc, val_macro_f1=vf1))
            print(f'{name} epoch {ep}/{epochs}  train loss {tl / len(d_train):.3f} acc {tc / len(d_train):.3f} | val loss {vloss:.3f} acc {vacc:.3f} macro-F1 {vf1:.3f}  ({time.time() - t0:.0f}s)')
            if vf1 > best:
                best, best_state = vf1, {k: v.detach().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)
    m.eval()
    return m, hist

def evaluate(P, y):
    pred = P.argmax(1)
    r = dict(acc=float((pred == y).mean()), macro_f1=float(f1_score(y, pred, average='macro')))
    for i, c in enumerate(CLASSES):
        r['recall_' + c] = float(((pred == y) & (y == i)).sum() / max(1, (y == i).sum()))
    return r

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d
print('helpers ready')

In [ ]:
# === 5. Evaluation sets, then the plain baselines (no regularization of any kind) ===
tr_all = df[df.split == 'train']
tr_real = tr_all[~tr_all.synthetic]
va = df[df.split == 'val']
SETS = {'val': va,
        'test': df[df.split == 'test'],                    # TrashNet photos, never trained on
        'external': df[df.split == 'test_external'],       # Garbage Dataset photos, never trained on
        'bgshift_val': df[df.split == 'val_bgshift'],      # same items on new backgrounds
        'bgshift': df[df.split == 'test_bgshift']}
EVL = {k: loader(v, eval_tf, False) for k, v in SETS.items()}
MODELS, PROBS, HIST = {}, {}, {}

def run_mode(name, arch, d_train, mode, **kw):
    m, h = train_model(arch, d_train, va, name=name, mode=mode, **kw)
    MODELS[name], HIST[name] = m, h
    PROBS[name] = {k: predict(m, dl) for k, dl in EVL.items()}

def ens(names, k):
    return np.mean([PROBS[n][k] for n in names], axis=0)

print('train real:', len(tr_real), '| train real + synthetic:', len(tr_all))
run_mode('B0_plain_real', 'efficientnet_b0', tr_real, 'plain')                  # plain model, real photos only
run_mode('B1_plain_real+synthetic', 'efficientnet_b0', tr_all, 'plain')         # plain model + synthetic images

In [ ]:
# === 6. Add dropout on its own, and check whether the plain model overfits ===
run_mode('D1_plain+dropout_real+syn', 'efficientnet_b0', tr_all, 'dropout')     # same data as B1, dropout added alone

def gap_table(names):
    rows = []
    for n in names:
        h = HIST[n]; b = max(h, key=lambda e: e['val_macro_f1']); l = h[-1]
        rows.append(dict(model=n, best_epoch=b['epoch'], train_acc=round(l['train_acc'], 3), val_acc=round(l['val_acc'], 3),
                         gap=round(l['train_acc'] - l['val_acc'], 3), train_loss=round(l['loss'], 3), val_loss=round(l['val_loss'], 3),
                         min_val_loss=round(min(e['val_loss'] for e in h), 3)))
    return pd.DataFrame(rows)

PLAIN = ['B0_plain_real', 'B1_plain_real+synthetic', 'D1_plain+dropout_real+syn']
pd.set_option('display.width', 220)
print(gap_table(PLAIN).to_string(index=False))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, n in zip(ax, PLAIN):
    h = HIST[n]
    a.plot([e['epoch'] for e in h], [e['loss'] for e in h], marker='o', label='train loss')
    a.plot([e['epoch'] for e in h], [e['val_loss'] for e in h], marker='o', label='validation loss')
    a.set_title(n, fontsize=9); a.set_xlabel('epoch'); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# === 7. A second backbone, then ensembles of the plain models ===
run_mode('B2_plain_mobilenetv3', 'mobilenet_v3_large', tr_all, 'plain')
ENS_DEF = {'ENS_plain(B1+B2)': ['B1_plain_real+synthetic', 'B2_plain_mobilenetv3'],
           'ENS_plain(B0+B1+B2)': ['B0_plain_real', 'B1_plain_real+synthetic', 'B2_plain_mobilenetv3']}
def add_ensembles():
    for nm, mem in ENS_DEF.items():
        PROBS[nm] = {k: ens(mem, k) for k in EVL}
add_ensembles()

def compact(names):
    rows = []
    for n in names:
        r = {'model': n}
        for k in ['test', 'external', 'bgshift']:
            e = evaluate(PROBS[n][k], SETS[k].label.values)
            r[k + '_F1'] = round(e['macro_f1'], 3); r[k + '_acc'] = round(e['acc'], 3)
            r[k + '_worst'] = round(min(e['recall_' + c] for c in CLASSES), 2)
        rows.append(r)
    return pd.DataFrame(rows)
print(compact(PLAIN + ['B2_plain_mobilenetv3'] + list(ENS_DEF)).to_string(index=False))

In [ ]:
# === 8. Later experiment: the full regularization stack (dropout, stochastic depth, label smoothing, weight decay, class weights, augmentation) ===
run_mode('E0_reg_real_only', 'efficientnet_b0', tr_real, 'reg')
run_mode('E1_reg_real+synthetic', 'efficientnet_b0', tr_all, 'reg')
run_mode('E2_reg_mobilenetv3', 'mobilenet_v3_large', tr_all, 'reg')
ENS_DEF.update({'ENS_reg(E1+E2)': ['E1_reg_real+synthetic', 'E2_reg_mobilenetv3'],
                'ENS_reg(E0+E1+E2)': ['E0_reg_real_only', 'E1_reg_real+synthetic', 'E2_reg_mobilenetv3']})
add_ensembles()
REG = ['E0_reg_real_only', 'E1_reg_real+synthetic', 'E2_reg_mobilenetv3']
print('(train loss below includes label smoothing, so it is not directly comparable with the plain models)')
print(gap_table(REG).to_string(index=False))
print(compact(REG + [n for n in ENS_DEF if 'reg' in n]).to_string(index=False))

In [ ]:
# === 9. Results table: every model on every evaluation set ===
rows = []
for n, d in PROBS.items():
    for k in ['test', 'external', 'bgshift']:
        rows.append(dict(model=n, set=k, **evaluate(d[k], SETS[k].label.values)))
res = pd.DataFrame(rows)
print('MACRO-F1'); print(res.pivot(index='model', columns='set', values='macro_f1').round(3))
print('ACCURACY'); print(res.pivot(index='model', columns='set', values='acc').round(3))
rc = [c for c in res.columns if c.startswith('recall_')]
for k in ['test', 'external', 'bgshift']:
    print('PER-CLASS RECALL on', k); print(res[res.set == k].set_index('model')[rc].round(2))

In [ ]:
# === 10. Final model (chosen on the validation set), published baseline, targets, and 95% intervals ===
yv, yb = SETS['val'].label.values, SETS['bgshift_val'].label.values
val_f1 = {n: float(f1_score(yv, PROBS[n]['val'].argmax(1), average='macro')) for n in PROBS}
bg_f1 = {n: float(f1_score(yb, PROBS[n]['bgshift_val'].argmax(1), average='macro')) for n in PROBS}
score = {n: (val_f1[n] + bg_f1[n]) / 2 for n in PROBS}
print('VALIDATION macro-F1: clean photos, new backgrounds, and the average of the two (chooses the final model; no test set is used)')
print(pd.DataFrame({'clean': val_f1, 'new_bg': bg_f1, 'average': score}).sort_values('average', ascending=False).round(3).to_string())
FINAL = max(score, key=score.get)
FINAL_MEMBERS = ENS_DEF.get(FINAL, [FINAL])
print('FINAL model:', FINAL, '->', FINAL_MEMBERS)

# Published baselines for the same datasets (accuracy). Edit/extend after reading the papers yourself.
LIT = {'TrashNet, pretrained GoogLeNet, random 10% hold-out (Yu & Grammenos, arXiv 2108.06274)': 0.912,
       'TrashNet, same model + augmentation (Yu & Grammenos)': 0.954,
       'Garbage Dataset, 10 classes, ResNet/EfficientNetV2 (GD paper, arXiv 2602.10500)': '0.93-0.96'}
for k, v in LIT.items():
    print(f'  published: {k}: {v}')
# Targets come from the baseline and the app, not from our own results. EDIT and defend them in the report.
TARGETS = dict(accuracy=0.91, worst_class_recall=0.75)
print('TARGETS', TARGETS)
for n in PROBS:
    for k in ['test', 'external']:
        e = evaluate(PROBS[n][k], SETS[k].label.values)
        worst = min(e['recall_' + c] for c in CLASSES)
        ok = e['acc'] >= TARGETS['accuracy'] and worst >= TARGETS['worst_class_recall']
        print(f'{n:26s} {k:9s} accuracy {e["acc"]:.3f}  worst-class recall {worst:.2f}  ->', 'MEETS TARGETS' if ok else 'below target')

print()
for k in ['test', 'external', 'bgshift']:
    y = SETS[k].label.values
    pred = PROBS[FINAL][k].argmax(1)
    lo, hi = wilson(int((pred == y).sum()), len(y))
    print(f'{FINAL} on {k:9s} n={len(y):3d}  accuracy {np.mean(pred == y):.3f}  95% CI [{lo:.3f}, {hi:.3f}]')
    for i, c in enumerate(CLASSES):
        n_c = int((y == i).sum()); k_c = int(((pred == i) & (y == i)).sum())
        lo, hi = wilson(k_c, n_c)
        print(f'     {c:10s} n={n_c:3d}  recall {k_c / n_c:.2f}  95% CI [{lo:.2f}, {hi:.2f}]')

In [ ]:
# === 11. Install a segmentation model (U2-Net, via rembg) ===
# After this cell finishes, run the next cell.
import subprocess
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rembg', 'onnxruntime'], capture_output=True, text=True).stdout[-300:])
print('installed')

In [ ]:
# === 12. Segment first, then classify (same models as the final model) ===
from rembg import new_session, remove
sess = new_session('u2netp')
N_PER_CLASS = 40          # subset per class so this step stays quick on Colab CPU

def isolate(path):
    im = get_img(path)
    rgba = remove(im, session=sess)
    alpha = np.array(rgba)[:, :, 3].astype(np.float32) / 255
    if alpha.mean() < 0.03 or alpha.mean() > 0.95:        # mask failed: keep the original photo
        return im, False
    canvas = Image.new('RGB', (W, H), (124, 124, 124))
    canvas.paste(rgba, mask=rgba.split()[3])
    return canvas, True

@torch.no_grad()
def probs_pil(models_, imgs, bs=64):
    out = []
    for i in range(0, len(imgs), bs):
        x = torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(DEVICE)
        out.append(np.mean([F.softmax(m(x), 1).float().cpu().numpy() for m in models_], axis=0))
    return np.concatenate(out)

members = [MODELS[n] for n in FINAL_MEMBERS]
seg_rows, shown = [], {}
for k in ['test', 'external', 'bgshift']:
    d = pd.concat([g.sample(min(len(g), N_PER_CLASS), random_state=0) for _, g in SETS[k].groupby('label')]).reset_index(drop=True)
    orig = [get_img(p) for p in d.path]
    iso = [isolate(p) for p in d.path]
    used = np.array([u for _, u in iso])
    y = d.label.values
    for tag, imgs in [('whole photo', orig), ('segment then classify', [i for i, _ in iso])]:
        r = evaluate(probs_pil(members, imgs), y)
        seg_rows.append(dict(set=k, input=tag, n=len(y), masks_used=f'{used.mean():.0%}', acc=r['acc'], macro_f1=r['macro_f1']))
    shown[k] = (orig[:3], [i for i, _ in iso][:3])
seg = pd.DataFrame(seg_rows)
print(seg.round(3).to_string(index=False))
fig, ax = plt.subplots(2, 6, figsize=(15, 4.4))
for j, k in enumerate(['external', 'bgshift']):
    for a in range(3):
        ax[0, j * 3 + a].imshow(shown[k][0][a]); ax[1, j * 3 + a].imshow(shown[k][1][a])
        ax[0, j * 3 + a].set_title(k, fontsize=8)
for a in ax.ravel():
    a.axis('off')
ax[0, 0].text(-10, 100, 'original', rotation=90, va='center'); ax[1, 0].text(-10, 100, 'segmented', rotation=90, va='center')
plt.tight_layout(); plt.show()

In [ ]:
# === 13. Plots for the report (also saved as PNG files) ===
os.makedirs('/content/out', exist_ok=True)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for n, h in HIST.items():
    ax[0].plot([e['epoch'] for e in h], [e['val_macro_f1'] for e in h], marker='o', label=n)
ax[0].set_title('Validation macro-F1 per epoch'); ax[0].set_xlabel('epoch'); ax[0].legend(fontsize=6)
for j, k in enumerate(['external', 'bgshift']):
    cm = confusion_matrix(SETS[k].label.values, PROBS[FINAL][k].argmax(1), labels=range(len(CLASSES)))
    ax[j + 1].imshow(cm, cmap='Blues')
    ax[j + 1].set_xticks(range(6)); ax[j + 1].set_xticklabels(CLASSES, rotation=45, ha='right')
    ax[j + 1].set_yticks(range(6)); ax[j + 1].set_yticklabels(CLASSES)
    for a in range(6):
        for b in range(6):
            ax[j + 1].text(b, a, cm[a, b], ha='center', va='center', fontsize=8)
    ax[j + 1].set_title(f'{FINAL} on {k}', fontsize=9); ax[j + 1].set_xlabel('predicted'); ax[j + 1].set_ylabel('true')
plt.tight_layout(); plt.savefig('/content/out/validation_and_confusion.png', dpi=150); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, n in zip(ax, PLAIN):
    h = HIST[n]
    a.plot([e['epoch'] for e in h], [e['loss'] for e in h], marker='o', label='train loss')
    a.plot([e['epoch'] for e in h], [e['val_loss'] for e in h], marker='o', label='validation loss')
    a.set_title(n, fontsize=9); a.set_xlabel('epoch'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig('/content/out/plain_train_vs_val_loss.png', dpi=150); plt.show()

In [ ]:
# === 14. Save models, tables and settings (the zip is only downloaded if you set DOWNLOAD = True) ===
DOWNLOAD = False
from google.colab import files
os.makedirs('/content/out', exist_ok=True)
for n, m in MODELS.items():
    torch.save(m.state_dict(), f'/content/out/{n}.pt')
res.to_csv('/content/out/results.csv', index=False)
try:
    seg.to_csv('/content/out/segmentation_results.csv', index=False)
except NameError:
    pass
json.dump({n: h for n, h in HIST.items()}, open('/content/out/history.json', 'w'))
json.dump(dict(final=FINAL, final_members=FINAL_MEMBERS, member_archs={n: ('mobilenet_v3_large' if 'mobilenet' in n else 'efficientnet_b0') for n in FINAL_MEMBERS},
               input_size=[H, W], mean=MEAN, std=STD, targets=TARGETS, classes=CLASSES), open('/content/out/config.json', 'w'))
shutil.make_archive('/content/final_project_outputs', 'zip', '/content/out')
if DOWNLOAD:
    files.download('/content/final_project_outputs.zip')
else:
    print('saved and zipped in /content/final_project_outputs.zip (not downloaded)')